In [7]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [8]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

In [9]:
# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    nome = f'df_ativos_{ano_usado}.csv'
    # caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv('../dados_retornos/otimizacao/ativos/'+nome)
    dfs[ano_usado]['date'] = pd.to_datetime(dfs[ano_usado]['date'])
    dfs[ano_usado] = dfs[ano_usado].set_index('date')
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0).all()])
    print(ano,'-',len(dfs[ano_usado].columns))
    

2025-12-31 - 78
2024-12-31 - 78
2023-12-31 - 78
2022-12-31 - 77
2021-12-31 - 77
2020-12-31 - 73
2019-12-31 - 73
2018-12-31 - 70
2017-12-31 - 70
2016-12-31 - 67
2015-12-31 - 62


In [10]:
dfs['2025'] 

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,
2025-10-01,-0.008268,-0.008518,-0.023326,0.005713,-0.019901,-0.023131,-0.007242,-0.019696,-0.016960,0.004511,...,0.007363,-0.001534,-0.024168,-0.034590,0.037825,0.012677,-0.017487,-0.005575,-0.021865,-0.013928
2025-10-02,-0.005011,-0.015227,-0.038790,-0.007953,-0.051440,-0.019096,-0.007752,-0.013595,-0.016177,-0.009879,...,-0.011643,-0.019191,-0.021195,-0.000474,-0.004556,0.006688,-0.013656,-0.018006,0.009500,-0.030610
2025-10-03,-0.005027,-0.000794,0.015509,-0.000191,-0.026400,-0.007016,-0.008274,0.000000,0.000587,-0.007857,...,0.005480,0.004695,0.018463,-0.010845,0.013730,-0.001874,0.005036,0.001204,0.007474,0.019431
2025-10-06,-0.001684,-0.018257,-0.012218,0.000955,-0.027483,-0.008626,-0.007878,-0.008158,-0.005846,-0.000612,...,-0.008992,-0.027658,-0.012308,-0.009065,0.033860,0.017068,-0.018791,-0.008406,-0.019781,0.001589
2025-10-07,-0.011810,-0.016977,-0.077401,-0.016208,-0.056893,-0.031641,-0.009341,-0.017822,-0.016474,0.007928,...,-0.010173,-0.042068,-0.016089,-0.010585,-0.034934,-0.014096,-0.019146,-0.022406,0.001402,-0.049167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-25,0.021086,0.014404,0.029187,0.018907,-0.020149,0.035675,0.008879,0.023927,0.017948,0.010737,...,0.015716,0.024595,0.005714,0.021536,0.014881,0.018566,0.028201,0.006577,0.015384,0.027903
2026-03-26,-0.011992,-0.022455,-0.016529,-0.030814,-0.043793,-0.012779,-0.033526,-0.013184,-0.023859,-0.005455,...,-0.019694,-0.052748,-0.010852,0.002811,-0.020528,-0.008045,-0.008828,-0.026599,-0.013699,-0.022767
2026-03-27,-0.004720,-0.013516,-0.036063,-0.016863,-0.029470,-0.031514,-0.017347,-0.014568,-0.015936,-0.006928,...,-0.016742,-0.019006,-0.010974,-0.007708,-0.013473,0.001141,-0.008678,-0.003737,-0.002314,0.000896


In [11]:
vb_cardinalidade_max = 10
vb_cardinalidade_min = 10
vb_peso_maximo = 0.20
vb_peso_minimo = 0.02
vb_theta = 0.5

## -----------------------

In [13]:
pd.read_excel(f'../score-raspagem/magic_formula/resultado/{ano_int}.xlsx')

,Unnamed: 0,ROIC,EY,magic_formula
0,USIM5,-3.651708,-0.056274,0.000000
1,BRAP4,-0.857439,-0.013075,0.065476
2,BBAS3,-0.762926,0.057898,0.069295
3,ITSA4,0.387140,0.013651,0.094813
4,AXIA3,0.716355,0.034243,0.102885
...,...,...,...,...
95,CVCB3,20.783002,0.404246,0.574474
96,WEGE3,22.843040,0.045096,0.613723
97,CURY3,26.523999,0.137918,0.700805
98,BBSE3,28.758551,0.073674,0.750886


In [14]:
y = []
carteiras_anuais = {}
melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])
print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)
lista_ativos_finais = {}
for an in anos:
    ano = an.split("-")[0]
    ano_int = int(ano)	
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass
    try:
        # retorno_usado = pd.DataFrame(eval(df_usado))
        df_usado = dfs[str(ano_int)]
        retorno_usado = df_usado.copy()

        score_todos_anos = pd.read_excel(f'../score-raspagem/magic_formula/resultado/{ano_int}.xlsx')
        score_usado_p = score_todos_anos[['Unnamed: 0', 'magic_formula']].set_index('Unnamed: 0').T
        
        ativos = retorno_usado.columns.intersection(score_usado_p.columns)
        retorno_usado = retorno_usado[ativos]
        score = score_usado_p[ativos]

        # COV
        df_cov = retorno_usado.cov()

        # Lista ativos
        lista_ativos_finais[ano] = retorno_usado.columns.tolist()
      
    except Exception as e:
            print("=========ERRRRRRRRRRRROR")
            print(e)    

    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",ano_int+1)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",ano)

    #  --------- MODELO
   

    #  --------- MODELO
    model = pyo.ConcreteModel()
    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns)
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns)-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize=lambda model,a,b: df_cov.iloc[a,b])
    # model.sigma_min = pyo.Param(initialize=0, mutable=True)
    # model.sigma_max = pyo.Param(initialize=1, mutable=True)
    model.score = pyo.Param(model.ativos, initialize=lambda model,a: score.iloc[0,a])
    model.theta = pyo.Param(initialize=0.5)

    # ------------------------------

    #linearizar Sharpe
    # z:= kx
    model.z = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
    model.k = pyo.Var(domain=pyo.NonNegativeReals)
    model.y = pyo.Var(model.ativos, domain=pyo.Binary)
    model.w = pyo.Var(model.ativos, domain=pyo.NonNegativeReals)
    model.KMAX = pyo.Param(initialize=100)

    # Com a problemática de cardinalidade y e bilinearidade, resolve-se
    # PESO
    # antes era model.x[a] <= model.pesomax * model.y[a]
    #  z=kx  -> model.z[a] <= model.pesomax * modely[a] * model.k[a]
    # CARDINALIDADE
    # era sum(model.y[a] ...) >= e <=
    # é feita a tranformação de w <= k * y

    # Com y=0 
    def res_y_zero(model,a):
            return model.w[a] <= model.KMAX * model.y[a]
    model.y_zero = pyo.Constraint(model.ativos, rule=res_y_zero)

    #--------------------------------------------------------------------------
    # com y=1
    #  segundo teto, quando y=1 ; w = k , pelas restriçoes >= e <=
    def res_y_1_teto2(model,a):
            return model.w[a] <= model.k 
    model.res_teto2 = pyo.Constraint(model.ativos, rule=res_y_1_teto2)

    # Limite inf 1
    def limite_1(model,a):
            return model.w[a] >= model.k - model.KMAX*(1-model.y[a])
    model.r_limite1 = pyo.Constraint(model.ativos, rule=limite_1)
    #--------------------------------------------------------------------------
    #  PESOS MIN MAX
    def pesomin(model,a):
            return model.z[a] >= 0.02 * model.w[a]
    model.r_pesomin = pyo.Constraint(model.ativos, rule=pesomin)

    def pesomax(model,a):
            return model.z[a] <= 0.2 * model.w[a]
    model.r_pesomax = pyo.Constraint(model.ativos, rule=pesomax)

    #  cardinalidade
    def card_min(model):
            return sum(model.y[a] for a in model.ativos) >= vb_cardinalidade_min
    model.r_cardmin = pyo.Constraint(rule=card_min)
    def card_max(model):
            return sum(model.y[a] for a in model.ativos) <= vb_cardinalidade_max
    model.r_card_max = pyo.Constraint(rule=card_max)

    # Restrições Sharpe
    def restricao_z(model):
            return sum(model.z[a] for a in model.ativos) == model.k
    model.r_restricao_z = pyo.Constraint(rule=restricao_z)

    def restricao_z_retorno(model):
            return sum(model.sigma[a,b]*model.z[a]*model.z[b] for a in model.ativos for b in model.ativos)  <= 1
    model.r_res_z_retorno = pyo.Constraint(rule=restricao_z_retorno)

    ## OBJETIVOS:  1 para pegar z min, 2 para z max, 3 para normalização
    def obj_1(model):
            var = sum(model.z[a] * model.retornos_ativos[d,a] for a in model.ativos for d in model.dias)
            score = sum(model.z[a] * model.score[a] for a in model.ativos)
            return var + score
    model.obj1 = pyo.Objective(rule=obj_1, sense=pyo.maximize)

    # ------------------- solver 1
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)
#     model.write('modelo_debug.lp', io_options={'symbolic_solver_labels': True})
   
    melhor_pesos = {list(model.nome_ativos.data())[a]:(model.z[a].value/model.k.value) for a in model.ativos}
    carteiras_anuais[int(ano_int)+1] = {
    	'pesos':  melhor_pesos,
    	# 'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano_int} -> {melhor_pesos}")
    print("+-="*30)
    if 'model' in locals():
    	del model
    	print('DELETADO DPS DO WHILE')
    else:
    	print("Nao consta model")
    	pass



=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2025
Modelo Antigo deletado 
 Iniciando Novo
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2026
## UTILIZANDO DADOS DE RETORNO DE:  2025
2025 -> {'ABEV3': 0.02613441324594628, 'ALOS3': 0.0, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.0, 'B3SA3': 0.0, 'BBAS3': 0.0, 'BBDC3': 0.0, 'BBDC4': 0.0, 'BBSE3': 0.199999998765681, 'BEEF3': 0.0, 'BPAC11': 0.0, 'BRAP4': 0.0, 'BRAV3': 0.0, 'CMIG4': 0.0, 'COGN3': 0.0, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.0, 'CSMG3': 0.09472850026457824, 'CSNA3': 0.0, 'CVCB3': 0.0, 'CXSE3': 0.0, 'CYRE3': 0.0, 'DIRR3': 0.0, 'ECOR3': 0.0, 'EGIE3': 0.0, 'EMBJ3': 0.0, 'ENEV3': 0.0, 'ENGI11': 0.0, 'EQTL3': 0.0, 'EZTC3': 0.0, 'FLRY3': 0.0, 'GGBR4': 0.0,

In [16]:
carteiras_anuais

{2026: {'pesos': {'ABEV3': 0.02613441324594628,
   'ALOS3': 0.0,
   'ANIM3': 0.0,
   'AXIA3': 0.0,
   'AZZA3': 0.0,
   'B3SA3': 0.0,
   'BBAS3': 0.0,
   'BBDC3': 0.0,
   'BBDC4': 0.0,
   'BBSE3': 0.199999998765681,
   'BEEF3': 0.0,
   'BPAC11': 0.0,
   'BRAP4': 0.0,
   'BRAV3': 0.0,
   'CMIG4': 0.0,
   'COGN3': 0.0,
   'CPFE3': 0.0,
   'CPLE3': 0.0,
   'CSAN3': 0.0,
   'CSMG3': 0.09472850026457824,
   'CSNA3': 0.0,
   'CVCB3': 0.0,
   'CXSE3': 0.0,
   'CYRE3': 0.0,
   'DIRR3': 0.0,
   'ECOR3': 0.0,
   'EGIE3': 0.0,
   'EMBJ3': 0.0,
   'ENEV3': 0.0,
   'ENGI11': 0.0,
   'EQTL3': 0.0,
   'EZTC3': 0.0,
   'FLRY3': 0.0,
   'GGBR4': 0.0,
   'GOAU4': 0.0,
   'HYPE3': 0.0,
   'IGTI11': 0.0,
   'IRBR3': 0.0,
   'ISAE4': 0.0,
   'ITSA4': 0.0,
   'ITUB3': 0.020000000573979382,
   'ITUB4': 0.0,
   'JHSF3': 0.020000001326470878,
   'KLBN11': 0.0,
   'LREN3': 0.0,
   'MDNE3': 0.0,
   'MGLU3': 0.0,
   'MOTV3': 0.0,
   'MOVI3': 0.020000000542858477,
   'MRVE3': 0.0,
   'MULT3': 0.0,
   'PETR3': 0.199

In [17]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])+1
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        # if v >= vb_peso_minimo and v <= vb_peso_maximo:
        if v >= vb_peso_minimo:
            linhas.append({'ano': an, 'ativo': k, 'peso': round(v, 4)})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2026
2025
2024
2023
2022
2021
2020
2019
2018
2017
2016


In [20]:
df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)['peso']

102    0.2000
103    0.1617
104    0.1581
100    0.1344
107    0.1023
101    0.0782
109    0.0515
105    0.0480
106    0.0440
108    0.0217
Name: peso, dtype: float64

In [ ]:
df_portfolios.to_csv('../carteiras/Sharpe_linear/magic_formula/sharpe_linear_magic_formula.csv')
